# Phase 3 (Transcription) on Colab T4

Runs `src/transcription/transcribe.py` (faster-whisper `large-v3`) against a T4 GPU instead of the local T1000 (4GB VRAM) box.

**Before running:**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Upload `data/raw/audio/` (and ideally `data/raw/metadata/`, for the duration sum used in the RTF extrapolation) to a folder in your Google Drive.
3. Edit `DRIVE_AUDIO_DIR` and `DRIVE_TRANSCRIPTS_DIR` in the **Configure paths** cell below to match where you put it.

Output lands directly in the Drive-mounted transcripts folder, so there's no separate download step -- it's already in the same place you'd sync back to local/T1000 from.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv

## 2. Clone the repo

In [ ]:
!git clone https://github.com/DAG-21/PureBillion-Cloner.git
%cd PureBillion-Cloner
!git log --oneline -5

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configure paths

Edit these two to match where you uploaded/want the data in your Drive, then run the cell.

In [ ]:
# EDIT THESE to match your Drive layout
DRIVE_AUDIO_DIR = "/content/drive/MyDrive/persona-clone/data/raw/audio"
DRIVE_METADATA_DIR = "/content/drive/MyDrive/persona-clone/data/raw/metadata"  # used for RTF extrapolation only
DRIVE_TRANSCRIPTS_DIR = "/content/drive/MyDrive/persona-clone/data/transcripts"

import os
assert os.path.isdir(DRIVE_AUDIO_DIR), f"Not found: {DRIVE_AUDIO_DIR} -- upload your audio there first"
os.makedirs(DRIVE_TRANSCRIPTS_DIR, exist_ok=True)
print("Audio files found:", len(os.listdir(DRIVE_AUDIO_DIR)))

## 5. Install dependencies

Only what Phase 3 needs -- same reasoning as on the T1000 box: don't install the full `requirements.txt` (it pulls in Linux-only `vllm` and deps for stages not being run here).

In [ ]:
!pip install -q faster-whisper pyyaml tqdm

In [ ]:
import ctranslate2
print("CUDA devices visible to ctranslate2:", ctranslate2.get_cuda_device_count())

## 6. Dry run

Confirms the file list without loading the model or touching the GPU -- same check that was done on the Dell laptop before the T1000 run.

In [ ]:
!python -m src.transcription.transcribe \
  --input-dir "{DRIVE_AUDIO_DIR}" \
  --output-dir "{DRIVE_TRANSCRIPTS_DIR}" \
  --dry-run

## 7. Single-file RTF benchmark

Transcribes one real file and times it, mirroring the planned T1000 benchmark step -- gives an apples-to-apples real-time-factor (RTF) number to compare T4 vs. T1000 before committing to a full run on either.

T4 has 16GB VRAM (vs. the T1000's 4GB), so this uses plain `float16` instead of the T1000's VRAM-constrained `int8_float16`.

In [ ]:
import glob, os, shutil, tempfile, time

audio_files = sorted(glob.glob(os.path.join(DRIVE_AUDIO_DIR, "*")))
assert audio_files, "No audio files found in DRIVE_AUDIO_DIR"
sample_file = audio_files[0]
print("Benchmarking on:", sample_file)

# Run the benchmark against a scratch dir so it doesn't affect the real
# transcription_history.csv / skip-if-exists logic for the full run below.
bench_input_dir = tempfile.mkdtemp()
bench_output_dir = tempfile.mkdtemp()
shutil.copy(sample_file, bench_input_dir)

In [ ]:
start = time.time()
!python -m src.transcription.transcribe \
  --input-dir "{bench_input_dir}" \
  --output-dir "{bench_output_dir}" \
  --history-file "{bench_output_dir}/bench_history.csv" \
  --device cuda \
  --compute-type float16
elapsed = time.time() - start
print(f"Wall clock: {elapsed:.1f}s")

In [ ]:
import json

transcript_path = glob.glob(os.path.join(bench_output_dir, "*.json"))[0]
with open(transcript_path) as f:
    transcript = json.load(f)

audio_duration = transcript["duration"]
rtf = elapsed / audio_duration
print(f"Audio duration: {audio_duration:.1f}s")
print(f"Transcription wall clock: {elapsed:.1f}s")
print(f"RTF (wall_clock / audio_duration): {rtf:.3f}")
print(f"-> roughly {1/rtf:.1f}x real-time on this T4")

In [ ]:
# Extrapolate to the full corpus, if metadata is available
if os.path.isdir(DRIVE_METADATA_DIR):
    total_duration = 0.0
    for meta_file in glob.glob(os.path.join(DRIVE_METADATA_DIR, "*.json")):
        with open(meta_file) as f:
            meta = json.load(f)
        total_duration += meta.get("duration", 0)
    est_seconds = total_duration * rtf
    print(f"Total corpus duration: {total_duration/3600:.1f} hours")
    print(f"Estimated full-batch transcription time on this T4: {est_seconds/3600:.1f} hours")
else:
    print(f"DRIVE_METADATA_DIR not found ({DRIVE_METADATA_DIR}) -- skipping full-corpus estimate.")

## 8. Full batch run

Only run this once the estimate above looks reasonable for a single Colab session (free tier disconnects on ~90 min idle and caps sessions around ~12h -- if the estimate exceeds that, either upgrade to Colab Pro or split the batch across multiple sessions, since the pipeline already skips already-transcribed IDs on rerun).

Writes straight into the Drive-mounted `DRIVE_TRANSCRIPTS_DIR`, so results persist even if the Colab runtime is later recycled.

In [ ]:
!python -m src.transcription.transcribe \
  --input-dir "{DRIVE_AUDIO_DIR}" \
  --output-dir "{DRIVE_TRANSCRIPTS_DIR}" \
  --device cuda \
  --compute-type float16

## 9. Sanity check the output

In [ ]:
import glob
n = len(glob.glob(os.path.join(DRIVE_TRANSCRIPTS_DIR, "*.json")))
print(f"{n} transcript JSON files in {DRIVE_TRANSCRIPTS_DIR}")
!head -c 500 "{glob.glob(os.path.join(DRIVE_TRANSCRIPTS_DIR, '*.json'))[0]}"

## Next steps

- Sync `DRIVE_TRANSCRIPTS_DIR` back down to local (or to the T1000 machine) so Phase 4 (diarization) has the transcripts to work from.
- `configs/transcription.yaml` in this repo clone is still set for the T1000 (`int8_float16`). This notebook overrides `--device`/`--compute-type` on the command line instead of editing the file, so nothing here needs committing back -- but note it if you ever want the config itself to default to T4 settings.
- Update `PROJECT_UPDATES.md` with the real RTF/timing numbers from step 7 once you have them.